# Aurum Market · Práctica de Bases de Datos Vectoriales

**Motor de descubrimiento y control de catálogo sobre una base de datos vectorial persistente (Qdrant).**

| Capacidad | Entrada | Salida | Criterio de aceptación (verificable en este notebook) |
|---|---|---|---|
| Búsqueda semántica | `query_text` (es) | top-k con `product_id`, `title`, `brand`, `score` (coseno, mayor = mejor) | Las 12 consultas ciegas devuelven 10 resultados válidos cada una |
| Búsqueda filtrada | `query_text` + filtro `brand equals X` | top-k donde **todos** los resultados cumplen el filtro | Las 4 consultas filtradas devuelven ≥1 resultado y el 100% pertenece a la marca; el filtro se ejecuta **en el motor**, no a posteriori |
| Mutaciones | 24 eventos (`UPSERT`/`DELETE`) en orden `sequence` | colección actualizada | Estado final verificado por ID y por búsqueda; repetir el lote no cambia nada (idempotencia) |
| Control de duplicados | ficha de alta (`title`, `brand`, `color`, `text`) | `decisión` + `candidato` + `score` | Regla calibrada en desarrollo (precision/recall/F1); decisión emitida para las 14 altas ciegas con candidatos procedentes de la base vectorial |
| Operativa | — | latencia, persistencia | p50/p95 medidos con repeticiones; el recuento sobrevive a una reconexión y el almacenamiento vive en un volumen Docker |

**Compromisos asumidos**: priorizamos un modelo multilingüe pequeño (coste de CPU contenido, 384 dims) frente a modelos grandes;
el catálogo completo (15.000 productos) se ingiere en Qdrant y toda la evaluación final se hace sobre él.
Las comparaciones de modelo/composición se hacen sobre la muestra de desarrollo para abaratar iteración, cambiando un único factor a la vez.

## 1. Arquitectura

```
CSV catálogo (15.000) ──> preparación de texto ──> embeddings (SentenceTransformers)
                                                        │
                       ┌────────────────────────────────┘
                       ▼
               Qdrant (Docker, volumen persistente ./qdrant_storage)
               · colección `aurum_catalogo` (384 dims, coseno, HNSW m=16, ef_construct=128)
               · payload: product_id, title, brand, brand_norm, color, locale, catalog_version, active
               · índices de payload: brand_norm (keyword), active (bool)
                       │
        ┌──────────────┼───────────────────┬──────────────────┐
        ▼              ▼                   ▼                  ▼
   búsqueda global  búsqueda filtrada   mutaciones        duplicados
   (query_points)   (Filter en motor)   (upsert/delete)   (vecinos + umbral + margen)
```

Requisito previo: `docker compose up -d` levanta Qdrant en `http://localhost:6333` (ver `docker-compose.yml` e `INSTALLATION.md`).

In [1]:
# === Configuración, versiones y semillas ====================================
from pathlib import Path
import json
import math
import os
import platform
import random
import sys
import time
import uuid
from collections import defaultdict
from importlib.metadata import version as pkg_version

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from dotenv import load_dotenv

pio.renderers.default = "notebook_connected"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / ".." / "data").resolve().exists() else Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.resolve()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
CACHE_DIR = PROJECT_ROOT / ".cache_embeddings"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from aurum.metricas import dcg, ndcg_at_k, recall_at_k, mrr_at_k, relevant_set
from aurum.texto import COMPOSICIONES, MAX_DESC_CHARS, build_product_text, normalize_brand

load_dotenv(PROJECT_ROOT / ".env")

# Modo humo (opcional): AURUM_SMOKE=1 ejecuta el recorrido completo con un
# subconjunto pequeño para validar el notebook en ~3 minutos.
SMOKE = os.environ.get("AURUM_SMOKE", "0") == "1"

CONFIG = {
    "seed": SEED,
    "top_k": 10,
    "modelos": {
        "minilm": {"nombre": "sentence-transformers/all-MiniLM-L6-v2", "prefijos": False,
                   "idiomas": "inglés", "dim": 384},
        "e5": {"nombre": "intfloat/multilingual-e5-small", "prefijos": True,
               "idiomas": "multilingüe (incluye español)", "dim": 384},
    },
    "qdrant_url": os.environ.get("QDRANT_URL", "http://localhost:6333"),
    "coleccion": "aurum_catalogo" + ("_smoke" if SMOKE else ""),
    "hnsw": {"m": 16, "ef_construct": 128},
    "batch_ingesta": 256,
    "latencia": {"calentamiento": 3, "repeticiones": 3 if SMOKE else 20},
}

ENTORNO = {
    "python": sys.version.split()[0],
    "plataforma": platform.platform(),
    "cpu": platform.processor(),
    "nucleos": os.cpu_count(),
    "paquetes": {p: pkg_version(p) for p in
                 ["sentence-transformers", "qdrant-client", "torch", "numpy", "pandas", "scikit-learn"]},
}
print(json.dumps(ENTORNO, indent=2, ensure_ascii=False))
print("Modo humo:", SMOKE, "| Colección:", CONFIG["coleccion"])

{
  "python": "3.14.3",
  "plataforma": "Windows-11-10.0.26200-SP0",
  "cpu": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
  "nucleos": 16,
  "paquetes": {
    "sentence-transformers": "5.7.0",
    "qdrant-client": "1.20.0",
    "torch": "2.15.0",
    "numpy": "2.5.4",
    "pandas": "3.1.0",
    "scikit-learn": "1.8.0"
  }
}
Modo humo: False | Colección: aurum_catalogo


In [2]:
# === Conexión a Qdrant (SDK nativo) =========================================
from qdrant_client import QdrantClient, models

def conectar_qdrant():
    '''Conecta al servidor Qdrant. Si no hay servidor, usa el modo local
    persistente del SDK (./qdrant_local) para que el notebook siga siendo
    reproducible sin Docker; el modo recomendado es el servidor.'''
    try:
        cli = QdrantClient(url=CONFIG["qdrant_url"], timeout=120)
        cli.get_collections()
        return cli, "servidor"
    except Exception as exc:
        print(f"Aviso: no hay servidor Qdrant en {CONFIG['qdrant_url']} ({type(exc).__name__}). "
              "Usando modo local persistente del SDK.")
        return QdrantClient(path=str(PROJECT_ROOT / "qdrant_local")), "local"

client, QDRANT_MODE = conectar_qdrant()
print("Conectado a Qdrant en modo:", QDRANT_MODE)

Conectado a Qdrant en modo: servidor


## 2. Carga de datos y contrato de esquema

Trabajamos con el **catálogo completo** (`catalogo_productos.csv`, 15.000 productos) como corpus del sistema.
La muestra de 1.500 (`catalogo_muestra.csv`) se usa solo para las comparaciones controladas de modelo y composición.
Validamos el contrato de datos con comprobaciones ejecutables: unicidad de `record_id` (el ID del punto vectorial),
cobertura de los juicios de relevancia y tipado de los metadatos.

In [3]:
catalog_full = pd.read_csv(DATA_DIR / "catalogo_productos.csv")
catalog_sample = pd.read_csv(DATA_DIR / "catalogo_muestra.csv")
queries_dev = pd.read_csv(DATA_DIR / "consultas_desarrollo.csv")
queries_eval = pd.read_csv(DATA_DIR / "consultas_evaluacion.csv")
queries_filtered = pd.read_csv(DATA_DIR / "consultas_filtradas.csv")
relevances_dev = pd.read_csv(DATA_DIR / "relevancias_desarrollo.csv")
altas_dev = pd.read_csv(DATA_DIR / "altas_desarrollo.csv")
altas_eval = pd.read_csv(DATA_DIR / "altas_evaluacion.csv")
events_catalog = pd.read_csv(DATA_DIR / "eventos_catalogo.csv")

if SMOKE:
    # En modo humo el corpus es la muestra (contiene el 100% de los juicios).
    catalog_full = catalog_sample.copy()

# --- Contrato de esquema (comprobaciones ejecutables) -----------------------
assert catalog_full["record_id"].is_unique, "record_id debe ser único (ID del punto)"
assert catalog_full["product_id"].is_unique, "product_id debe ser único"
assert catalog_full["title"].notna().all(), "title no puede ser nulo"
assert set(relevances_dev["product_id"]).issubset(set(catalog_full["product_id"])), \
    "todos los productos juzgados deben estar en el corpus"
assert relevances_dev["relevance"].isin([0, 1, 2, 3]).all()
assert set(events_catalog["operation"].unique()) <= {"UPSERT", "DELETE"}

print("catalogo (corpus):", catalog_full.shape)
print("muestra desarrollo:", catalog_sample.shape)
print("consultas dev/eval/filtradas:", len(queries_dev), len(queries_eval), len(queries_filtered))
print("juicios de relevancia:", len(relevances_dev))
print("altas dev/eval:", len(altas_dev), len(altas_eval))
print("eventos:", len(events_catalog), dict(events_catalog["operation"].value_counts()))
print("nulos por columna:", catalog_full.isna().sum().to_dict())

catalogo (corpus): (15000, 9)
muestra desarrollo: (1500, 9)
consultas dev/eval/filtradas: 8 12 4
juicios de relevancia: 248
altas dev/eval: 14 14
eventos: 24 {'UPSERT': np.int64(16), 'DELETE': np.int64(8)}
nulos por columna: {'record_id': 0, 'product_id': 0, 'title': 0, 'brand': 658, 'color': 5609, 'locale': 0, 'text': 0, 'catalog_version': 0, 'active': 0}


## 3. Contrato de relevancia y métricas (corregidas)

Etiquetas ESCI → relevancia graduada: `E`=3, `S`=2, `C`=1, `I`=0.

**Correcciones respecto a la entrega anterior** (señaladas en la evaluación):

1. **nDCG@10**: antes el DCG ideal se calculaba solo con los documentos *recuperados*, lo que regalaba un nDCG=1 a
   cualquier ranking que ordenara bien lo poco que trajera. Ahora el IDCG usa **todos los juicios de la consulta**:
   no recuperar los productos más relevantes penaliza.
2. **Recall@10 y MRR@10**: antes el conjunto de referencia incluía todos los productos *juzgados*, incluso los
   irrelevantes (`I`=0). Ahora solo cuentan los **relevantes** (relevancia > 0).

Las funciones viven en [`src/aurum/metricas.py`](../src/aurum/metricas.py) y están cubiertas por tests
(`tests/test_metricas.py`). Abajo se validan con un ejemplo calculado a mano.

In [4]:
# --- Validación numérica de las métricas con un ejemplo a mano --------------
juicios = {"A": 3, "B": 2, "C": 0, "D": 1}   # juzgados: A,B,D relevantes; C irrelevante
ranking = ["C", "A", "X"]                     # X no juzgado

# DCG = (2^0-1)/log2(2) + (2^3-1)/log2(3) + (2^0-1)/log2(4) = 7/log2(3)
# IDCG = (2^3-1)/log2(2) + (2^2-1)/log2(3) + (2^1-1)/log2(4) = 7 + 3/log2(3) + 0.5
esperado_ndcg = (7 / math.log2(3)) / (7 + 3 / math.log2(3) + 0.5)
assert abs(ndcg_at_k(ranking, juicios, k=10) - esperado_ndcg) < 1e-12
assert recall_at_k(ranking, juicios, k=10) == 1 / 3      # solo A de {A,B,D}
assert mrr_at_k(ranking, juicios, k=10) == 1 / 2         # primer relevante en pos. 2 (C no cuenta)
assert ndcg_at_k(["A", "B", "D"], juicios, k=10) == 1.0  # ranking perfecto

# El error antiguo habría dado: recall = 1/4 (contando C) y ndcg con ideal solo de lo recuperado.
print("Métricas corregidas validadas ✔")
print(f"nDCG ejemplo = {ndcg_at_k(ranking, juicios):.4f} (esperado {esperado_ndcg:.4f})")

QRELS_DEV = {
    int(qid): dict(zip(grp["product_id"].astype(str), grp["relevance"].astype(int)))
    for qid, grp in relevances_dev.groupby("query_id")
}

def evaluar_run(run, qrels, k=10):
    '''run: {query_id: [product_id ordenados]} -> DataFrame con métricas por consulta.'''
    filas = []
    for qid, ranked in run.items():
        j = qrels.get(qid, {})
        filas.append({"query_id": qid,
                      "ndcg@10": ndcg_at_k(ranked, j, k),
                      "recall@10": recall_at_k(ranked, j, k),
                      "mrr@10": mrr_at_k(ranked, j, k)})
    return pd.DataFrame(filas).sort_values("query_id").reset_index(drop=True)

label_counts = (relevances_dev["relevance"].map({3: "E", 2: "S", 1: "C", 0: "I"})
                .value_counts().reindex(["E", "S", "C", "I"]).reset_index())
label_counts.columns = ["etiqueta", "pares"]
fig = px.bar(label_counts, x="etiqueta", y="pares", text="pares",
             title="Distribución de juicios ESCI en desarrollo (248 pares)")
fig.show()

Métricas corregidas validadas ✔
nDCG ejemplo = 0.4702 (esperado 0.4702)


## 4. Baseline léxico (TF-IDF)

Referencia reproducible contra la que se compara toda configuración densa. **Protocolo común a todas las
comparaciones de este notebook**: mismo corpus, mismas 8 consultas de desarrollo, mismos juicios, mismo k=10
y mismas métricas corregidas. El baseline usa la misma composición de texto que el sistema denso (título +
atributos + descripción truncada) para que la única diferencia sea la representación.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

textos_corpus = catalog_full.apply(lambda r: build_product_text(r, "c3_completo"), axis=1).tolist()
ids_corpus = catalog_full["product_id"].astype(str).tolist()

t0 = time.perf_counter()
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(textos_corpus)
print(f"TF-IDF sobre corpus completo: {X_tfidf.shape} en {time.perf_counter()-t0:.1f}s")

def buscar_lexico(query_text, top_k=10):
    q = tfidf.transform([query_text])
    scores = linear_kernel(q, X_tfidf).ravel()
    idx = np.argsort(-scores)[:top_k]
    return [ids_corpus[i] for i in idx]

run_lexico = {int(r["query_id"]): buscar_lexico(r["query_text"]) for _, r in queries_dev.iterrows()}
res_lexico = evaluar_run(run_lexico, QRELS_DEV)
print("\nBaseline léxico (corpus completo, métricas corregidas):")
print(res_lexico[["ndcg@10", "recall@10", "mrr@10"]].mean().round(4).to_dict())
res_lexico

TF-IDF sobre corpus completo: (15000, 50000) en 1.9s

Baseline léxico (corpus completo, métricas corregidas):
{'ndcg@10': 0.4028, 'recall@10': 0.1705, 'mrr@10': 0.7656}


,query_id,ndcg@10,recall@10,mrr@10
0,13357,0.571121,0.193548,1.000
1,18868,0.256265,0.111111,1.000
2,28703,0.224006,0.051282,0.500
3,31224,0.348597,0.187500,0.500
4,33633,0.095162,0.333333,0.125
5,38249,0.466642,0.111111,1.000
6,43240,0.504549,0.142857,1.000
7,61533,0.756448,0.233333,1.000


## 5. Representación densa

### 5.1 Preparación del texto: composiciones y control de longitud

Comparamos tres composiciones (definidas en [`src/aurum/texto.py`](../src/aurum/texto.py)), cambiando solo ese factor:

- **c1_titulo**: solo `title`.
- **c2_titulo_atributos**: `title` + `Marca:` + `Color:`.
- **c3_completo**: c2 + descripción **truncada a 600 caracteres** (los modelos truncan en 256-512 tokens;
  sin control de longitud, la parte final de descripciones de hasta 3.000 caracteres se descarta de forma opaca).

El contrato de preparación garantiza que los nulos **no** se incorporan como texto (no aparece `nan` ni
`Marca:` vacío) y está probado en `tests/test_texto.py`.

In [6]:
longitud = catalog_full["text"].astype(str).str.len()
print("Longitud de la descripción original:",
      {"p25": int(longitud.quantile(.25)), "p50": int(longitud.quantile(.5)),
       "p75": int(longitud.quantile(.75)), "max": int(longitud.max())})

textos_por_composicion = {
    comp: catalog_sample.apply(lambda r: build_product_text(r, comp), axis=1).tolist()
    for comp in COMPOSICIONES
}
for comp, txts in textos_por_composicion.items():
    arr = pd.Series([len(t) for t in txts])
    print(f"{comp:22s} len p50={int(arr.median()):5d}  p95={int(arr.quantile(.95)):5d}  max={int(arr.max()):5d}")

ejemplo = catalog_sample.iloc[4]
for comp in COMPOSICIONES:
    print(f"\n--- {comp} ---\n{build_product_text(ejemplo, comp)[:300]}")

Longitud de la descripción original: {'p25': 321, 'p50': 935, 'p75': 2007, 'max': 3000}
c1_titulo              len p50=  126  p95=  197  max=  225
c2_titulo_atributos    len p50=  159  p95=  229  max=  288
c3_completo            len p50=  742  p95=  832  max=  891

--- c1_titulo ---
COSTWAY Mesa de Ordenador Escritorio Esquina en Forma de L Estación de Trabajo para Hogar Oficina (Marrón)

--- c2_titulo_atributos ---
COSTWAY Mesa de Ordenador Escritorio Esquina en Forma de L Estación de Trabajo para Hogar Oficina (Marrón) | Marca: COSTWAY | Color: Marrón

--- c3_completo ---
COSTWAY Mesa de Ordenador Escritorio Esquina en Forma de L Estación de Trabajo para Hogar Oficina (Marrón) | Marca: COSTWAY | Color: Marrón | COSTWAY Mesa de Ordenador Escritorio Esquina en Forma de L Estación de Trabajo para Hogar Oficina (Marrón). Marca: COSTWAY. Color: Marrón. Características: 【E


### 5.2 Selección del modelo de embeddings (comparación controlada)

El catálogo y las consultas están en **español**. Comparamos dos configuraciones con el mismo coste de
dimensión (384) y protocolos idénticos (muestra de 1.500, composición `c3_completo` fija, búsqueda **exacta**
con NumPy para aislar la representación de cualquier efecto del índice ANN):

| Clave | Modelo | Idiomas | Contrato de codificación |
|---|---|---|---|
| `minilm` | `sentence-transformers/all-MiniLM-L6-v2` | inglés | sin prefijos |
| `e5` | `intfloat/multilingual-e5-small` | multilingüe | **requiere** prefijos `query:` / `passage:` |

Ambos se usan con `normalize_embeddings=True` (vectores unitarios), versión fijada en `requirements.txt`,
y medimos también el **coste de codificación** para que la decisión considere calidad y coste.

In [7]:
from sentence_transformers import SentenceTransformer

MODELOS = {}
def obtener_modelo(clave):
    if clave not in MODELOS:
        MODELOS[clave] = SentenceTransformer(CONFIG["modelos"][clave]["nombre"])
    return MODELOS[clave]

def codificar(clave, textos, tipo, batch_size=64):
    '''Codifica respetando el contrato de cada modelo (prefijos de E5) y
    devolviendo vectores float32 normalizados (norma 1).'''
    assert tipo in ("query", "passage")
    if CONFIG["modelos"][clave]["prefijos"]:
        prefijo = "query: " if tipo == "query" else "passage: "
        textos = [prefijo + t for t in textos]
    emb = obtener_modelo(clave).encode(textos, batch_size=batch_size, convert_to_numpy=True,
                                       normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(emb, dtype=np.float32)

def codificar_con_cache(clave, textos, tipo, etiqueta):
    ruta = CACHE_DIR / f"{etiqueta}_{clave}_{tipo}_{len(textos)}.npy"
    if ruta.exists():
        emb = np.load(ruta)
        if emb.shape[0] == len(textos):
            return emb, 0.0
    t0 = time.perf_counter()
    emb = codificar(clave, textos, tipo)
    dt = time.perf_counter() - t0
    np.save(ruta, emb)
    return emb, dt

def busqueda_exacta_numpy(q_emb, doc_embs, ids, top_k=10):
    scores = doc_embs @ q_emb
    idx = np.argsort(-scores)[:top_k]
    return [ids[i] for i in idx], scores[idx]

ids_muestra = catalog_sample["product_id"].astype(str).tolist()
textos_dev = queries_dev["query_text"].tolist()
qids_dev = queries_dev["query_id"].astype(int).tolist()

comparacion_modelos = []
emb_muestra = {}
for clave in CONFIG["modelos"]:
    docs, t_docs = codificar_con_cache(clave, textos_por_composicion["c3_completo"], "passage",
                                       "muestra_c3")
    emb_muestra[clave] = docs
    q_embs = codificar(clave, textos_dev, "query")
    run = {qid: busqueda_exacta_numpy(q_embs[i], docs, ids_muestra)[0]
           for i, qid in enumerate(qids_dev)}
    met = evaluar_run(run, QRELS_DEV)[["ndcg@10", "recall@10", "mrr@10"]].mean()
    comparacion_modelos.append({"modelo": clave, "nombre": CONFIG["modelos"][clave]["nombre"],
                                **met.round(4).to_dict(),
                                "coste_codificacion_s_1500_docs": round(t_docs, 1)})

tabla_modelos = pd.DataFrame(comparacion_modelos)
MODELO_GANADOR = tabla_modelos.sort_values("ndcg@10", ascending=False).iloc[0]["modelo"]
print(f"Modelo elegido por nDCG@10 en desarrollo: {MODELO_GANADOR}")
tabla_modelos

C:\Users\diego\Documents\PontIA\BDVextoriales\BDVectorial-practica\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3825.84it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3599.19it/s]

Modelo elegido por nDCG@10 en desarrollo: e5


,modelo,nombre,ndcg@10,recall@10,mrr@10,coste_codificacion_s_1500_docs
0,minilm,sentence-transformers/all-MiniLM-L6-v2,0.4820,0.2744,0.7812,0.0
1,e5,intfloat/multilingual-e5-small,0.7151,0.3159,0.9062,0.0


**Lectura**: el modelo multilingüe parte con ventaja estructural en un corpus en español: `all-MiniLM-L6-v2`
se entrenó con pares en inglés y trata el español como vocabulario parcialmente desconocido. La tabla anterior
cuantifica esa diferencia con el mismo protocolo; el coste de codificación acompaña a la decisión
(E5-small es ~1,7× más lento por tener 12 capas frente a 6, un compromiso aceptable para ingesta por lotes).

### 5.3 Comparación de composiciones de texto (con el modelo elegido)

In [8]:
comparacion_composiciones = []
for comp in COMPOSICIONES:
    docs, t_docs = codificar_con_cache(MODELO_GANADOR, textos_por_composicion[comp], "passage",
                                       f"muestra_{comp}")
    q_embs = codificar(MODELO_GANADOR, textos_dev, "query")
    run = {qid: busqueda_exacta_numpy(q_embs[i], docs, ids_muestra)[0]
           for i, qid in enumerate(qids_dev)}
    met = evaluar_run(run, QRELS_DEV)[["ndcg@10", "recall@10", "mrr@10"]].mean()
    comparacion_composiciones.append({"composicion": comp, **met.round(4).to_dict()})

tabla_composiciones = pd.DataFrame(comparacion_composiciones)
COMPOSICION_GANADORA = tabla_composiciones.sort_values("ndcg@10", ascending=False).iloc[0]["composicion"]
print(f"Composición elegida: {COMPOSICION_GANADORA}")
tabla_composiciones

Composición elegida: c3_completo


,composicion,ndcg@10,recall@10,mrr@10
0,c1_titulo,0.6024,0.2898,0.7750
1,c2_titulo_atributos,0.6986,0.3497,0.8438
2,c3_completo,0.7151,0.3159,0.9062


### 5.4 Geometría, métrica y semántica del score

- Los vectores se normalizan (norma 1), por lo que **producto escalar = similitud coseno**.
- Qdrant se configura con `Distance.COSINE` y devuelve **similitud** en `[-1, 1]`: **mayor = mejor**.
- Comprobamos numéricamente la equivalencia y que la normalización se cumple.

In [9]:
docs = emb_muestra[MODELO_GANADOR]
normas = np.linalg.norm(docs, axis=1)
assert np.allclose(normas, 1.0, atol=1e-5), "los vectores deben ser unitarios"

q = codificar(MODELO_GANADOR, ["zapatillas cómodas para correr"], "query")[0]
punto = docs[0]
coseno = float(np.dot(q, punto) / (np.linalg.norm(q) * np.linalg.norm(punto)))
escalar = float(np.dot(q, punto))
assert abs(coseno - escalar) < 1e-6
print(f"norma media de documentos: {normas.mean():.6f}")
print(f"producto escalar = {escalar:.6f}  |  coseno = {coseno:.6f}  -> equivalentes ✔")
print("Semántica del score: similitud coseno, rango [-1, 1], mayor = mejor; se conserva el score nativo del motor.")

norma media de documentos: 1.000000
producto escalar = 0.806431  |  coseno = 0.806431  -> equivalentes ✔
Semántica del score: similitud coseno, rango [-1, 1], mayor = mejor; se conserva el score nativo del motor.


## 6. Base de datos vectorial: esquema, ingesta y persistencia (Qdrant)

**Modelo lógico de la colección `aurum_catalogo`:**

| Elemento | Valor | Justificación |
|---|---|---|
| ID del punto | `record_id` (UUIDv5 estable del dataset) | upserts idempotentes: reingestar no duplica |
| Vector | 384 dims, `Distance.COSINE` | vectores unitarios del modelo elegido |
| Índice | HNSW `m=16`, `ef_construct=128` (explícitos) | compromiso estándar memoria/recall; fidelidad medida en §8 |
| Payload | `product_id`, `title`, `brand`, `brand_norm`, `color`, `locale`, `catalog_version`, `active` | filtros en el motor + trazabilidad de versiones |
| Índices de payload | `brand_norm` (keyword), `active` (bool) | filtrado eficiente por marca y visibilidad |

La ingesta es **por lotes** (256 puntos), con `wait=True`, verificación de recuento y estado, reingesta
idempotente demostrada y persistencia verificada tras reconectar el cliente (el almacenamiento vive en el
volumen Docker `./qdrant_storage`, de modo que sobrevive a reinicios del contenedor).

In [10]:
COLECCION = CONFIG["coleccion"]
DIM = CONFIG["modelos"][MODELO_GANADOR]["dim"]

# Limpieza protegida: solo se borra/recrea la colección con este nombre exacto.
if client.collection_exists(COLECCION):
    client.delete_collection(COLECCION)

client.create_collection(
    collection_name=COLECCION,
    vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
    hnsw_config=models.HnswConfigDiff(**CONFIG["hnsw"]),
    # Umbral bajo para forzar la construcción del índice HNSW con 15k puntos.
    optimizers_config=models.OptimizersConfigDiff(indexing_threshold=1000),
)
client.create_payload_index(COLECCION, field_name="brand_norm",
                            field_schema=models.PayloadSchemaType.KEYWORD)
client.create_payload_index(COLECCION, field_name="active",
                            field_schema=models.PayloadSchemaType.BOOL)

info = client.get_collection(COLECCION)
print("Colección creada:", COLECCION)
print("  dim:", info.config.params.vectors.size, "| métrica:", info.config.params.vectors.distance)
print("  HNSW m:", info.config.hnsw_config.m, "| ef_construct:", info.config.hnsw_config.ef_construct)

Colección creada: aurum_catalogo
  dim: 384 | métrica: Cosine
  HNSW m: 16 | ef_construct: 128


In [11]:
# --- Embeddings del corpus completo con la configuración ganadora -----------
textos_ingesta = catalog_full.apply(lambda r: build_product_text(r, COMPOSICION_GANADORA), axis=1).tolist()
emb_corpus, t_embed_corpus = codificar_con_cache(
    MODELO_GANADOR, textos_ingesta, "passage", f"corpus_{COMPOSICION_GANADORA}")
print(f"Embeddings del corpus: {emb_corpus.shape} "
      f"({'cache' if t_embed_corpus == 0 else f'{t_embed_corpus:.0f}s'})")

def payload_producto(row):
    return {
        "product_id": str(row["product_id"]),
        "title": None if pd.isna(row["title"]) else str(row["title"]),
        "brand": None if pd.isna(row["brand"]) else str(row["brand"]),
        "brand_norm": normalize_brand(row["brand"]),
        "color": None if pd.isna(row["color"]) else str(row["color"]),
        "locale": str(row["locale"]),
        "catalog_version": int(row["catalog_version"]),
        "active": bool(row["active"]),
    }

def ingerir(df, embeddings, batch=None):
    batch = batch or CONFIG["batch_ingesta"]
    filas = df.reset_index(drop=True)
    for inicio in range(0, len(filas), batch):
        bloque = filas.iloc[inicio:inicio + batch]
        puntos = [models.PointStruct(id=str(r["record_id"]),
                                     vector=embeddings[inicio + j].tolist(),
                                     payload=payload_producto(r))
                  for j, (_, r) in enumerate(bloque.iterrows())]
        client.upsert(COLECCION, points=puntos, wait=True)
    return len(filas)

t0 = time.perf_counter()
n_ingeridos = ingerir(catalog_full, emb_corpus)
t_ingesta = time.perf_counter() - t0

recuento = client.count(COLECCION, exact=True).count
print(f"Ingesta por lotes de {n_ingeridos} puntos en {t_ingesta:.1f}s "
      f"({CONFIG['batch_ingesta']} puntos/lote)")
assert recuento == len(catalog_full), f"recuento {recuento} != {len(catalog_full)}"
print(f"Recuento verificado en el motor: {recuento} ✔")

# Esperar a que el índice HNSW esté construido (estado green).
for _ in range(120):
    estado = client.get_collection(COLECCION).status
    if str(estado).endswith("GREEN") or str(estado) == "green":
        break
    time.sleep(1)
print("Estado de la colección:", client.get_collection(COLECCION).status)

Embeddings del corpus: (15000, 384) (531s)


Ingesta por lotes de 15000 puntos en 11.4s (256 puntos/lote)
Recuento verificado en el motor: 15000 ✔


Estado de la colección: green


In [12]:
# --- Idempotencia de la ingesta y persistencia tras reconexión ---------------
punto_antes = client.retrieve(COLECCION, ids=[str(catalog_full.iloc[0]["record_id"])], with_payload=True)[0]

ingerir(catalog_full.head(500), emb_corpus[:500])  # repetir parte de la ingesta
recuento_tras_repetir = client.count(COLECCION, exact=True).count
assert recuento_tras_repetir == len(catalog_full), "reingestar no debe duplicar"

punto_despues = client.retrieve(COLECCION, ids=[punto_antes.id], with_payload=True)[0]
assert punto_antes.payload == punto_despues.payload, "el payload debe ser estable"
print(f"Reingesta idempotente: recuento sigue siendo {recuento_tras_repetir} y el payload no cambia ✔")

# Persistencia: cerramos el cliente, reconectamos y el estado sigue ahí.
client.close()
client, _ = conectar_qdrant()
recuento_reconexion = client.count(COLECCION, exact=True).count
assert recuento_reconexion == len(catalog_full)
print(f"Tras reconectar el cliente, el recuento persiste: {recuento_reconexion} ✔")
print("El almacenamiento vive en ./qdrant_storage (volumen Docker): sobrevive a reinicios del contenedor.")

Reingesta idempotente: recuento sigue siendo 15000 y el payload no cambia ✔
Tras reconectar el cliente, el recuento persiste: 15000 ✔
El almacenamiento vive en ./qdrant_storage (volumen Docker): sobrevive a reinicios del contenedor.


## 7. Búsqueda y filtros ejecutados en el motor

Contrato uniforme de respuesta: lista de resultados con `rank`, `record_id`, `product_id`, `title`, `brand`,
`score` (similitud coseno, mayor = mejor). El filtro de marca se pasa a Qdrant como condición de payload
(`brand_norm == normalize_brand(valor)`), de modo que el motor devuelve exactamente `top_k` resultados **que ya
cumplen el filtro** — no se recupera un top global y se filtra después en Python (error de la entrega anterior,
que dejaba las 4 consultas filtradas vacías).

La ausencia de resultados se trata de forma explícita (lista vacía + aviso), no como un éxito silencioso.

In [13]:
EF_PRODUCCION = 128  # justificado en §8 con la medición de fidelidad

def aurum_search(query_text, top_k=10, brand=None, hnsw_ef=None, exact=False):
    '''Búsqueda semántica con filtro opcional de marca ejecutado en el motor.'''
    q_emb = codificar(MODELO_GANADOR, [query_text], "query")[0]
    condiciones = [models.FieldCondition(key="active", match=models.MatchValue(value=True))]
    if brand is not None:
        valor = normalize_brand(brand)
        condiciones.append(models.FieldCondition(key="brand_norm",
                                                 match=models.MatchValue(value=valor)))
    respuesta = client.query_points(
        COLECCION,
        query=q_emb.tolist(),
        limit=top_k,
        query_filter=models.Filter(must=condiciones),
        search_params=models.SearchParams(exact=exact,
                                          hnsw_ef=None if exact else (hnsw_ef or EF_PRODUCCION)),
        with_payload=True,
    )
    hits = [{"rank": i + 1, "record_id": str(p.id), "product_id": p.payload["product_id"],
             "title": p.payload["title"], "brand": p.payload["brand"], "score": float(p.score)}
            for i, p in enumerate(respuesta.points)]
    if not hits:
        print(f"Aviso: sin resultados para {query_text!r} (brand={brand!r}). "
              "Revisa el filtro o el estado de la colección.")
    return hits

pd.DataFrame(aurum_search("zapatillas cómodas para correr", top_k=5))

,rank,record_id,product_id,title,brand,score
0,1,15232d6d-e4fc-57db-b638-37ea11443a38,B01N2L4KEM,"Asics Patriot 8 W, Zapatillas De Running Mujer...",ASICS,0.903735
1,2,45df87eb-89d1-58b3-87a9-47dfae5f3e5a,B08XWJ3YW9,Zapatillas De Deporte Hombres Running Correr T...,ZMBCYG,0.902306
2,3,4ae09a1a-aafa-5dee-b371-3c88e0ecd454,B08YVLVKFD,Air Zapatillas de Running para Hombre Mujer De...,AONETIGER,0.888778
3,4,23426dd5-1833-570b-bdef-5a45c52fd1c5,B08Q8RF8LN,"Nike Run Swift 2, Zapatillas para Correr Hombr...",NIKE,0.888556
4,5,22b7989e-96db-54e2-a2b0-a57c567d1960,B08D8S41H1,Zapatos Deporte Mujer Hombre Zapatillas y Calz...,Sisttke,0.888407


In [14]:
# --- Validación de las 4 consultas filtradas (criterio de aceptación) --------
filas_filtro = []
for _, q in queries_filtered.iterrows():
    en_coleccion = client.count(COLECCION, exact=True, count_filter=models.Filter(must=[
        models.FieldCondition(key="brand_norm",
                              match=models.MatchValue(value=normalize_brand(q["filter_value"])))])).count
    hits = aurum_search(q["query_text"], top_k=10, brand=q["filter_value"])
    marcas = {h["brand"] for h in hits}
    if en_coleccion == 0:
        # Solo puede ocurrir en modo humo (corpus reducido); con el catálogo completo las 4 marcas existen.
        print(f"{q['workload_id']}: la marca {q['filter_value']!r} no existe en el corpus reducido; se omite.")
        continue
    assert len(hits) > 0, f"{q['workload_id']}: el filtro no puede dejar la consulta vacía"
    assert all(normalize_brand(h["brand"]) == normalize_brand(q["filter_value"]) for h in hits), \
        f"{q['workload_id']}: todos los resultados deben ser de la marca"
    filas_filtro.append({"workload_id": q["workload_id"], "filtro": q["filter_value"],
                         "en_coleccion": en_coleccion,
                         "resultados": len(hits), "marcas_devueltas": sorted(marcas),
                         "mejor_resultado": hits[0]["title"][:70], "score_top1": round(hits[0]["score"], 4)})
print("Consultas filtradas validadas: resultados no vacíos y 100% de la marca pedida ✔")
tabla_filtros = pd.DataFrame(filas_filtro)
tabla_filtros

Consultas filtradas validadas: resultados no vacíos y 100% de la marca pedida ✔


,workload_id,filtro,en_coleccion,resultados,marcas_devueltas,mejor_resultado,score_top1
0,FILTER-001,Einhell,30,10,[Einhell],"Einhell Martillo perforador TC-RH 900 (900 W, ...",0.8773
1,FILTER-002,Apple,100,10,[Apple],Apple iPad Mini 16GB 3G 4G Negro - Tablet (Min...,0.8257
2,FILTER-003,NIKE,295,10,[NIKE],"Nike MD Runner 2 (GS), Zapatillas de Correr Mu...",0.8818
3,FILTER-004,SAMSUNG,155,10,[SAMSUNG],"Samsung S24F354 - Monitor de 24"" Full HD (1920...",0.8494


In [15]:
# --- Casos límite: filtro sin coincidencias y selectividad -------------------
sin_resultados = aurum_search("zapatillas de correr", top_k=5, brand="MarcaInexistente")
assert sin_resultados == []

conteo_marca = {}
for marca in queries_filtered["filter_value"]:
    n = client.count(COLECCION, exact=True,
                     count_filter=models.Filter(must=[models.FieldCondition(
                         key="brand_norm", match=models.MatchValue(value=normalize_brand(marca)))])).count
    conteo_marca[marca] = n
print("Productos por marca en la colección (selectividad del filtro):", conteo_marca)

Aviso: sin resultados para 'zapatillas de correr' (brand='MarcaInexistente'). Revisa el filtro o el estado de la colección.
Productos por marca en la colección (selectividad del filtro): {'Einhell': 30, 'Apple': 100, 'NIKE': 295, 'SAMSUNG': 155}


## 8. Fidelidad del índice ANN frente al oráculo exacto

El mismo motor permite búsqueda **exacta** (`SearchParams(exact=True)`), que usamos como oráculo para medir
cuánta calidad pierde el índice HNSW según `hnsw_ef` (tamaño de la cola de candidatos en consulta).
Protocolo: 20 consultas (8 de desarrollo + 12 ciegas), solapamiento del top-10 (`recall@10` frente al oráculo)
y latencia p50 por configuración (5 repeticiones tras calentamiento).

In [16]:
consultas_fidelidad = queries_dev["query_text"].tolist() + queries_eval["query_text"].tolist()
vectores_fidelidad = codificar(MODELO_GANADOR, consultas_fidelidad, "query")

def ids_busqueda(q_emb, exact=False, hnsw_ef=None, top_k=10):
    resp = client.query_points(COLECCION, query=q_emb.tolist(), limit=top_k,
                               search_params=models.SearchParams(exact=exact, hnsw_ef=hnsw_ef),
                               with_payload=False)
    return [str(p.id) for p in resp.points]

oraculo = [ids_busqueda(v, exact=True) for v in vectores_fidelidad]

filas_fidelidad, solapamientos_por_consulta = [], {}
for ef in [4, 16, 64, 128, 256]:
    solapamientos, tiempos = [], []
    for _ in range(2):  # calentamiento
        ids_busqueda(vectores_fidelidad[0], hnsw_ef=ef)
    for rep in range(5):
        por_consulta = []
        for i, v in enumerate(vectores_fidelidad):
            t0 = time.perf_counter()
            ids_ann = ids_busqueda(v, hnsw_ef=ef)
            tiempos.append((time.perf_counter() - t0) * 1000)
            por_consulta.append(len(set(ids_ann) & set(oraculo[i])) / len(oraculo[i]))
        solapamientos.append(por_consulta)
    media_por_consulta = np.mean(solapamientos, axis=0)
    solapamientos_por_consulta[ef] = media_por_consulta
    filas_fidelidad.append({
        "hnsw_ef": ef,
        "recall@10 vs oráculo (medio)": round(float(np.mean(media_por_consulta)), 4),
        "recall@10 (peor consulta)": round(float(np.min(media_por_consulta)), 4),
        "latencia p50 (ms)": round(float(np.percentile(tiempos, 50)), 2),
        "latencia p95 (ms)": round(float(np.percentile(tiempos, 95)), 2),
    })

tabla_fidelidad = pd.DataFrame(filas_fidelidad)
tabla_fidelidad

,hnsw_ef,recall@10 vs oráculo (medio),recall@10 (peor consulta),latencia p50 (ms),latencia p95 (ms)
0,4,1.0,1.0,14.73,25.13
1,16,1.0,1.0,15.45,26.82
2,64,1.0,1.0,15.36,27.16
3,128,1.0,1.0,15.53,26.82
4,256,1.0,1.0,5.25,26.84


In [17]:
fig = px.line(tabla_fidelidad, x="latencia p50 (ms)", y="recall@10 vs oráculo (medio)",
              text="hnsw_ef", markers=True,
              title="Compromiso fidelidad-latencia del índice HNSW (oráculo = búsqueda exacta)")
fig.update_traces(textposition="top center")
fig.show()

peor_ef = tabla_fidelidad.iloc[0]
candidatos_ef = tabla_fidelidad[tabla_fidelidad["recall@10 vs oráculo (medio)"] >= 0.99]
ef_minimo_fiel = int(candidatos_ef.iloc[0]["hnsw_ef"]) if len(candidatos_ef) else 256
print(f"Primer hnsw_ef con recall medio >= 0.99 frente al oráculo: {ef_minimo_fiel}.")
print(f"Producción usa hnsw_ef={EF_PRODUCCION}: mantiene fidelidad ~1.0 con margen para búsquedas "
      "filtradas (el filtro reduce los candidatos útiles del grafo) a un coste de latencia asumible (tabla).")
consultas_sensibles = [(consultas_fidelidad[i], round(float(s), 3))
                       for i, s in enumerate(solapamientos_por_consulta[4]) if s < 1.0]
print(f"Consultas sensibles al ANN con ef=4 ({len(consultas_sensibles)}):")
for texto, s in consultas_sensibles[:5]:
    print(f"  recall={s}  {texto[:80]}")

Primer hnsw_ef con recall medio >= 0.99 frente al oráculo: 4.
Producción usa hnsw_ef=128: mantiene fidelidad ~1.0 con margen para búsquedas filtradas (el filtro reduce los candidatos útiles del grafo) a un coste de latencia asumible (tabla).
Consultas sensibles al ANN con ef=4 (0):


## 9. Evaluación comparativa sobre el catálogo completo

Comparación final (corrige la limitación de la entrega anterior, que evaluaba sobre la muestra de 1.500):
**TF-IDF vs denso**, ambos sobre los **15.000 productos**, mismas consultas, mismos juicios, métricas corregidas.
El sistema denso se evalúa a través del motor con la configuración de producción (`hnsw_ef=128`), que según §8
es fiel al oráculo.

In [18]:
run_denso = {}
for _, q in queries_dev.iterrows():
    hits = aurum_search(q["query_text"], top_k=10)
    run_denso[int(q["query_id"])] = [h["product_id"] for h in hits]

res_denso = evaluar_run(run_denso, QRELS_DEV)
resumen = pd.DataFrame({
    "TF-IDF (léxico)": res_lexico[["ndcg@10", "recall@10", "mrr@10"]].mean(),
    f"Denso ({MODELO_GANADOR} + Qdrant)": res_denso[["ndcg@10", "recall@10", "mrr@10"]].mean(),
}).round(4)
print(resumen)

detalle = res_lexico.merge(res_denso, on="query_id", suffixes=("_lexico", "_denso"))
detalle = detalle.merge(queries_dev[["query_id", "query_text"]], on="query_id")
largo = detalle.melt(id_vars=["query_id", "query_text"],
                     value_vars=["ndcg@10_lexico", "ndcg@10_denso"],
                     var_name="sistema", value_name="ndcg@10")
fig = px.bar(largo, x="query_id", y="ndcg@10", color="sistema", barmode="group",
             hover_data=["query_text"], title="nDCG@10 por consulta de desarrollo (corpus completo)")
fig.show()
detalle[["query_id", "query_text", "ndcg@10_lexico", "ndcg@10_denso",
         "recall@10_lexico", "recall@10_denso"]].round(3)

           TF-IDF (léxico)  Denso (e5 + Qdrant)
ndcg@10             0.4028               0.5625
recall@10           0.1705               0.2215
mrr@10              0.7656               0.7500


,query_id,query_text,ndcg@10_lexico,ndcg@10_denso,recall@10_lexico,recall@10_denso
0,13357,base tapizada 160x200 sin patas,0.571,0.553,0.194,0.258
1,18868,botines marrones mujer tacon medio,0.256,0.513,0.111,0.444
2,28703,convertibles 2 en 1 portátil tactil,0.224,0.753,0.051,0.179
3,31224,cámaras bridge baratas,0.349,0.331,0.188,0.188
4,33633,disfraz halloween talla grande hombre,0.095,0.000,0.333,0.000
5,38249,estantes sin taladro habitacion,0.467,0.428,0.111,0.083
6,43240,funda ipad air 4 sin tapa,0.505,0.922,0.143,0.286
7,61533,lentejas sin gluten,0.756,1.000,0.233,0.333


## 10. Metodología de latencia

Protocolo explícito (corrige la medición anterior, que guardaba una media de lote como p50/p95):

- 3 repeticiones de **calentamiento** (descartadas) + **20 repeticiones medidas** × 8 consultas = 160 muestras.
- Separamos **coste de codificación de la consulta** (CPU, modelo) y **coste de búsqueda** (motor + red local).
- p50/p95 calculados con `np.percentile` sobre las muestras individuales.
- Entorno y versiones registrados en la celda de configuración y exportados a `metricas_desarrollo.json`.

In [19]:
lat_embed, lat_search, lat_total = [], [], []
reps = CONFIG["latencia"]["repeticiones"]
warmup = CONFIG["latencia"]["calentamiento"]

for rep in range(warmup + reps):
    for texto in queries_dev["query_text"]:
        t0 = time.perf_counter()
        q_emb = codificar(MODELO_GANADOR, [texto], "query")[0]
        t1 = time.perf_counter()
        client.query_points(COLECCION, query=q_emb.tolist(), limit=10,
                            search_params=models.SearchParams(hnsw_ef=EF_PRODUCCION),
                            with_payload=True)
        t2 = time.perf_counter()
        if rep >= warmup:
            lat_embed.append((t1 - t0) * 1000)
            lat_search.append((t2 - t1) * 1000)
            lat_total.append((t2 - t0) * 1000)

def percentiles(muestras):
    return {"p50_ms": round(float(np.percentile(muestras, 50)), 2),
            "p95_ms": round(float(np.percentile(muestras, 95)), 2),
            "n_muestras": len(muestras)}

LATENCIA = {"protocolo": {"calentamiento": warmup, "repeticiones": reps,
                          "consultas": len(queries_dev), "hnsw_ef": EF_PRODUCCION},
            "codificacion_consulta": percentiles(lat_embed),
            "busqueda_motor": percentiles(lat_search),
            "total": percentiles(lat_total)}
print(json.dumps(LATENCIA, indent=2, ensure_ascii=False))

{
  "protocolo": {
    "calentamiento": 3,
    "repeticiones": 20,
    "consultas": 8,
    "hnsw_ef": 128
  },
  "codificacion_consulta": {
    "p50_ms": 9.77,
    "p95_ms": 13.25,
    "n_muestras": 160
  },
  "busqueda_motor": {
    "p50_ms": 6.46,
    "p95_ms": 24.15,
    "n_muestras": 160
  },
  "total": {
    "p50_ms": 19.49,
    "p95_ms": 33.69,
    "n_muestras": 160
  }
}


## 11. Detección de altas duplicadas

Los candidatos se obtienen **de la base vectorial sobre el catálogo completo** (corrige la entrega anterior,
que los sacaba de FAISS sobre la muestra). Regla: duplicado si `score_top1 >= umbral` **y**
`margen (top1 - top2) >= margen_min`. Calibramos ambos parámetros en desarrollo maximizando F1 y reportamos
precision/recall/F1; después aplicamos la regla congelada a las 14 altas ciegas.

In [20]:
def candidatos_alta(row, top_k=5):
    texto = build_product_text(row, COMPOSICION_GANADORA)
    # Ficha de producto entrante: se codifica como 'passage' (comparación documento-documento).
    q_emb = codificar(MODELO_GANADOR, [texto], "passage")[0]
    resp = client.query_points(COLECCION, query=q_emb.tolist(), limit=top_k,
                               search_params=models.SearchParams(hnsw_ef=256), with_payload=True)
    return [(p.payload["product_id"], float(p.score)) for p in resp.points]

filas = []
for _, alta in altas_dev.iterrows():
    cands = candidatos_alta(alta)
    filas.append({"incoming_id": alta["incoming_id"],
                  "candidato": cands[0][0], "score_top1": cands[0][1],
                  "score_top2": cands[1][1] if len(cands) > 1 else 0.0,
                  "margen": cands[0][1] - (cands[1][1] if len(cands) > 1 else 0.0),
                  "es_duplicado": bool(alta["is_duplicate"]),
                  "referencia": alta["reference_product_id"]})
dev_dup = pd.DataFrame(filas)
acierto_candidato = (dev_dup.loc[dev_dup["es_duplicado"], "candidato"]
                     == dev_dup.loc[dev_dup["es_duplicado"], "referencia"]).mean()
print(f"Candidato top-1 correcto en los duplicados reales de desarrollo: {acierto_candidato:.0%}")
dev_dup.round(4)

Candidato top-1 correcto en los duplicados reales de desarrollo: 100%


,incoming_id,candidato,score_top1,score_top2,margen,es_duplicado,referencia
0,DEV-DUP-001,B000G3T55M,0.9976,0.9956,0.0020,True,B000G3T55M
1,DEV-DUP-002,B07NV4L2W5,0.9749,0.9733,0.0016,True,B07NV4L2W5
2,DEV-DUP-003,B00BEFAR80,0.9831,0.9018,0.0813,True,B00BEFAR80
3,DEV-DUP-004,B076HKFZ8N,0.9934,0.9044,0.0890,True,B076HKFZ8N
4,DEV-DUP-005,B07JYHSK27,0.9914,0.8710,0.1203,True,B07JYHSK27
5,DEV-DUP-006,B07N379P73,0.9621,0.8852,0.0769,True,B07N379P73
6,DEV-DUP-007,B077FZDNJ2,0.9881,0.9676,0.0205,True,B077FZDNJ2
7,DEV-NEW-001,B07S2G3ZQH,0.8793,0.8692,0.0102,False,NaN
8,DEV-NEW-002,B08KSVGWYK,0.8823,0.8797,0.0026,False,NaN
9,DEV-NEW-003,B06VVVQ72B,0.8825,0.8612,0.0214,False,NaN


In [21]:
# --- Calibración de umbral y margen en desarrollo ----------------------------
umbrales = np.round(np.linspace(dev_dup["score_top1"].min() - 0.01,
                                dev_dup["score_top1"].max() + 0.001, 40), 4)
margenes = np.round(np.linspace(0.0, max(dev_dup["margen"].max(), 0.01), 20), 4)

mejor = None
for u in umbrales:
    for m in margenes:
        pred = (dev_dup["score_top1"] >= u) & (dev_dup["margen"] >= m)
        real = dev_dup["es_duplicado"]
        tp = int((pred & real).sum()); fp = int((pred & ~real).sum()); fn = int((~pred & real).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        # Desempate por precisión: un falso positivo fusiona fichas distintas
        # (coste asimétrico mayor que revisar manualmente un duplicado no detectado).
        if mejor is None or (f1, prec) > (mejor["f1"], mejor["precision"]):
            mejor = {"umbral": float(u), "margen_min": float(m),
                     "precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4)}

REGLA_DUP = mejor
print("Regla calibrada en desarrollo:", REGLA_DUP)

pred_final = (dev_dup["score_top1"] >= REGLA_DUP["umbral"]) & (dev_dup["margen"] >= REGLA_DUP["margen_min"])
errores = dev_dup[pred_final != dev_dup["es_duplicado"]]
print(f"Errores en desarrollo con la regla: {len(errores)}")
errores.round(4) if len(errores) else print("Sin errores de decisión en desarrollo ✔")

Regla calibrada en desarrollo:

 {'umbral': 0.9093, 'margen_min': 0.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Errores en desarrollo con la regla: 0
Sin errores de decisión en desarrollo ✔


In [22]:
# --- Aplicación de la regla congelada a las 14 altas ciegas ------------------
filas_eval_dup = []
for _, alta in altas_eval.iterrows():
    cands = candidatos_alta(alta)
    s1 = cands[0][1]; s2 = cands[1][1] if len(cands) > 1 else 0.0
    es_dup = bool(s1 >= REGLA_DUP["umbral"] and (s1 - s2) >= REGLA_DUP["margen_min"])
    filas_eval_dup.append({"incoming_id": alta["incoming_id"],
                           "predicted_duplicate": str(es_dup).lower(),
                           "matched_product_id": cands[0][0] if es_dup else "",
                           "score": round(s1, 6)})
duplicados_eval = pd.DataFrame(filas_eval_dup)
assert len(duplicados_eval) == len(altas_eval)
duplicados_eval

,incoming_id,predicted_duplicate,matched_product_id,score
0,EVAL-DUP-001,true,B081JP8CC6,0.979864
1,EVAL-DUP-002,true,B07GWRF23V,0.974586
2,EVAL-DUP-003,true,B07S7B3SN2,0.986262
3,EVAL-DUP-004,true,8417441271,0.956890
4,EVAL-DUP-005,true,B00JOH9FRO,0.969783
5,EVAL-DUP-006,true,B01JADTEKE,0.986443
6,EVAL-DUP-007,true,B01M0G3WP0,0.987457
7,EVAL-NEW-001,false,,0.868599
8,EVAL-NEW-002,false,,0.882079
9,EVAL-NEW-003,false,,0.876221


## 12. Exportación de artefactos (contrato de resultados)

Tres artefactos en `results/`, generados desde la base vectorial sobre el catálogo completo y validados
automáticamente (número de filas, columnas, orden de scores):

- `resultados_busqueda.csv`: top-10 de las 12 consultas ciegas (`evaluation_id, rank, product_id, score`).
- `resultados_duplicados.csv`: decisión por alta (`incoming_id, predicted_duplicate, matched_product_id, score`).
- `metricas_desarrollo.json`: métricas, latencia con percentiles reales, fidelidad ANN, comparaciones y protocolo.

Los artefactos se exportan **antes** de aplicar los eventos de catálogo (§13) para que reflejen el estado del
catálogo base sobre el que están definidos los juicios de evaluación.

In [23]:
filas_busqueda = []
for _, q in queries_eval.iterrows():
    hits = aurum_search(q["query_text"], top_k=10)
    for h in hits:
        filas_busqueda.append({"evaluation_id": q["evaluation_id"], "rank": h["rank"],
                               "product_id": h["product_id"], "score": round(h["score"], 6)})
df_busqueda = pd.DataFrame(filas_busqueda)

# Validación automática del contrato
assert len(df_busqueda) == len(queries_eval) * 10, "cada consulta ciega debe aportar 10 filas"
assert list(df_busqueda.columns) == ["evaluation_id", "rank", "product_id", "score"]
for eid, grp in df_busqueda.groupby("evaluation_id"):
    assert list(grp["rank"]) == list(range(1, 11))
    assert (grp["score"].diff().dropna() <= 1e-9).all(), "scores descendentes"
    assert grp["product_id"].isin(catalog_full["product_id"]).all()

df_busqueda.to_csv(RESULTS_DIR / "resultados_busqueda.csv", index=False)
duplicados_eval.to_csv(RESULTS_DIR / "resultados_duplicados.csv", index=False)
print("resultados_busqueda.csv y resultados_duplicados.csv validados y exportados ✔")

resultados_busqueda.csv y resultados_duplicados.csv validados y exportados ✔


In [24]:
metricas = {
    "lexical": {"ndcg_at_10": float(res_lexico["ndcg@10"].mean()),
                "recall_at_10": float(res_lexico["recall@10"].mean()),
                "mrr_at_10": float(res_lexico["mrr@10"].mean())},
    "dense": {"ndcg_at_10": float(res_denso["ndcg@10"].mean()),
              "recall_at_10": float(res_denso["recall@10"].mean()),
              "mrr_at_10": float(res_denso["mrr@10"].mean())},
    "latency_p50_ms": LATENCIA["total"]["p50_ms"],
    "latency_p95_ms": LATENCIA["total"]["p95_ms"],
    "latencia_detalle": LATENCIA,
    "fidelidad_ann": tabla_fidelidad.to_dict(orient="records"),
    "comparacion_modelos": tabla_modelos.to_dict(orient="records"),
    "comparacion_composiciones": tabla_composiciones.to_dict(orient="records"),
    "duplicados_regla": REGLA_DUP,
    "protocolo": {
        "corpus": int(len(catalog_full)), "consultas_desarrollo": int(len(queries_dev)),
        "k": 10, "seed": SEED,
        "modelo": CONFIG["modelos"][MODELO_GANADOR]["nombre"],
        "composicion_texto": COMPOSICION_GANADORA,
        "motor": f"Qdrant ({QDRANT_MODE})", "coleccion": COLECCION,
        "hnsw": {**CONFIG["hnsw"], "ef_busqueda": EF_PRODUCCION},
        "entorno": ENTORNO,
    },
}
with open(RESULTS_DIR / "metricas_desarrollo.json", "w", encoding="utf-8") as fh:
    json.dump(metricas, fh, indent=2, ensure_ascii=False)
print("metricas_desarrollo.json exportado ✔")
print(json.dumps({k: metricas[k] for k in ["lexical", "dense", "latency_p50_ms", "latency_p95_ms"]},
                 indent=2))

metricas_desarrollo.json exportado ✔
{
  "lexical": {
    "ndcg_at_10": 0.40284864343251775,
    "recall_at_10": 0.17050955876560714,
    "mrr_at_10": 0.765625
  },
  "dense": {
    "ndcg_at_10": 0.5624739618178256,
    "recall_at_10": 0.22148463655520106,
    "mrr_at_10": 0.75
  },
  "latency_p50_ms": 19.49,
  "latency_p95_ms": 33.69
}


## 13. Mutaciones: eventos de catálogo sobre la colección

Aplicamos los **24 eventos** de `eventos_catalogo.csv` en orden `sequence` **directamente sobre la colección
Qdrant** (corrige la entrega anterior, que solo mutaba un DataFrame):

- `UPSERT`: se reconstruye el texto con la composición vigente, se recodifica el vector y se upserta el punto
  con su payload (incluida `catalog_version`, que da trazabilidad de la revisión).
- `DELETE`: se borra **solo** el punto con ese `record_id` (nunca borrados por filtro amplio).

Verificamos cada operación por **ID** (retrieve) y la **visibilidad en el buscador** (el producto revisado
aparece al buscar su título; el borrado desaparece). Después repetimos el lote completo para demostrar
**idempotencia** en el motor.

In [25]:
def aplicar_eventos(eventos):
    '''Aplica los eventos en orden sobre la colección. Devuelve métricas de visibilidad.'''
    registro = []
    for _, ev in eventos.sort_values("sequence").iterrows():
        rid = str(ev["record_id"])
        t0 = time.perf_counter()
        if ev["operation"] == "UPSERT":
            texto = build_product_text(ev, COMPOSICION_GANADORA)
            vec = codificar(MODELO_GANADOR, [texto], "passage")[0]
            client.upsert(COLECCION, points=[models.PointStruct(
                id=rid, vector=vec.tolist(), payload=payload_producto(ev))], wait=True)
            visible = len(client.retrieve(COLECCION, ids=[rid])) == 1
        else:  # DELETE protegido: solo por ID explícito
            client.delete(COLECCION, points_selector=models.PointIdsList(points=[rid]), wait=True)
            visible = len(client.retrieve(COLECCION, ids=[rid])) == 0
        registro.append({"sequence": int(ev["sequence"]), "event_id": ev["event_id"],
                         "operation": ev["operation"], "record_id": rid,
                         "efecto_visible": visible,
                         "visibilidad_ms": round((time.perf_counter() - t0) * 1000, 1)})
    return pd.DataFrame(registro)

recuento_inicial = client.count(COLECCION, exact=True).count
registro_eventos = aplicar_eventos(events_catalog)
assert registro_eventos["efecto_visible"].all(), "cada operación debe ser visible tras aplicarse"

# Estado final esperado: replay de los eventos sobre el conjunto de IDs inicial.
ids_iniciales = set(catalog_full["record_id"].astype(str))
estado = set(ids_iniciales)
for _, ev in events_catalog.sort_values("sequence").iterrows():
    rid = str(ev["record_id"])
    estado.add(rid) if ev["operation"] == "UPSERT" else estado.discard(rid)
recuento_final = client.count(COLECCION, exact=True).count
assert recuento_final == len(estado), f"recuento {recuento_final} != esperado {len(estado)}"
print(f"Recuento: {recuento_inicial} -> {recuento_final} (esperado {len(estado)}) ✔")
print(f"Visibilidad por operación (ms): p50={registro_eventos['visibilidad_ms'].median():.0f}, "
      f"max={registro_eventos['visibilidad_ms'].max():.0f} (upsert/delete con wait=True son síncronos)")
registro_eventos.head(8)

Recuento: 15000 -> 15000 (esperado 15000) ✔
Visibilidad por operación (ms): p50=46, max=80 (upsert/delete con wait=True son síncronos)


,sequence,event_id,operation,record_id,efecto_visible,visibilidad_ms
0,1,EVT-001,UPSERT,e1a0e559-6a49-5be5-b617-ec8a4899e975,True,60.3
1,2,EVT-002,UPSERT,0df8a596-0bc2-5deb-bc84-69b63972f975,True,79.5
2,3,EVT-003,UPSERT,f5726a4e-1d31-5c25-b109-2f3fb8aa6567,True,78.8
3,4,EVT-004,UPSERT,35c0c946-4061-50f6-a672-96e62991f53b,True,68.8
4,5,EVT-005,UPSERT,7b2d843d-0607-54a8-863f-934716fda42e,True,39.9
5,6,EVT-006,UPSERT,e28de4d9-bb45-5a32-b163-4484c54f5bcf,True,79.8
6,7,EVT-007,UPSERT,1666e8ac-f215-5cbb-90be-e04feeff44a8,True,62.8
7,8,EVT-008,UPSERT,e5a3a7fb-8fb3-5e91-9c54-814075d63ee6,True,58.0


In [26]:
# --- Verificación por ID y por búsqueda --------------------------------------
ultimo_estado = {}
for _, ev in events_catalog.sort_values("sequence").iterrows():
    ultimo_estado[str(ev["record_id"])] = ev

upsertados = [rid for rid, ev in ultimo_estado.items() if ev["operation"] == "UPSERT"]
borrados = [rid for rid, ev in ultimo_estado.items() if ev["operation"] == "DELETE"]

for rid in upsertados:
    punto = client.retrieve(COLECCION, ids=[rid], with_payload=True)
    assert len(punto) == 1 and punto[0].payload["catalog_version"] == int(ultimo_estado[rid]["catalog_version"])
assert all(client.retrieve(COLECCION, ids=[rid]) == [] for rid in borrados)
print(f"Verificado por ID: {len(upsertados)} upserts con catalog_version correcta, "
      f"{len(borrados)} borrados ausentes ✔")

# Visibilidad en el buscador: el producto revisado se encuentra buscando su título.
muestra_upsert = ultimo_estado[upsertados[0]]
hits = aurum_search(str(muestra_upsert["title"])[:80], top_k=5)
assert any(h["record_id"] == upsertados[0] for h in hits), "el upsert debe ser visible en búsqueda"
print(f"Visible en búsqueda: {hits[0]['title'][:70]}... (catalog_version actualizada)")

if borrados:
    # Los eventos DELETE no llevan título: lo buscamos en el catálogo original.
    titulo_borrado = catalog_full.loc[catalog_full["record_id"].astype(str) == borrados[0], "title"]
    if len(titulo_borrado):
        hits_borrado = aurum_search(str(titulo_borrado.iloc[0])[:80], top_k=10)
        assert all(h["record_id"] != borrados[0] for h in hits_borrado)
        print("El registro borrado ya no aparece en la búsqueda ✔")

Verificado por ID: 16 upserts con catalog_version correcta, 8 borrados ausentes ✔
Visible en búsqueda: NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black... (catalog_version actualizada)
El registro borrado ya no aparece en la búsqueda ✔


In [27]:
# --- Idempotencia: repetir el lote completo no cambia nada -------------------
payloads_antes = {rid: client.retrieve(COLECCION, ids=[rid], with_payload=True)[0].payload
                  for rid in upsertados}
registro_segunda = aplicar_eventos(events_catalog)
recuento_segunda = client.count(COLECCION, exact=True).count
payloads_despues = {rid: client.retrieve(COLECCION, ids=[rid], with_payload=True)[0].payload
                    for rid in upsertados}

assert recuento_segunda == recuento_final, "repetir el lote no debe cambiar el recuento"
assert payloads_antes == payloads_despues, "repetir el lote no debe cambiar los payloads"
print(f"Idempotencia verificada en el motor: recuento estable ({recuento_segunda}) "
      "y payloads idénticos tras repetir los 24 eventos ✔")

Idempotencia verificada en el motor: recuento estable (15000) y payloads idénticos tras repetir los 24 eventos ✔


## 14. Atribución de errores y limitaciones

Analizamos fallos concretos con evidencia reproducible y los asignamos a una capa del sistema:
**representación** (modelo/texto), **índice ANN**, **datos/filtros** o **consistencia**.

In [28]:
# === Caso 1 (representación): la peor consulta de desarrollo =================
peor = res_denso.sort_values("ndcg@10").iloc[0]
qid_peor = int(peor["query_id"])
texto_peor = queries_dev.loc[queries_dev["query_id"] == qid_peor, "query_text"].iloc[0]
print(f"Peor consulta densa: {qid_peor} · {texto_peor!r} · nDCG@10={peor['ndcg@10']:.3f}")

hits = aurum_search(texto_peor, top_k=10)
juicios = QRELS_DEV[qid_peor]
tabla_caso1 = pd.DataFrame([{"rank": h["rank"], "product_id": h["product_id"],
                             "title": h["title"][:70], "score": round(h["score"], 4),
                             "relevancia": juicios.get(h["product_id"], "no juzgado")} for h in hits])
print(tabla_caso1.to_string(index=False))

# ¿Es culpa del ANN? Comparamos con el oráculo exacto: si devuelve lo mismo, la
# pérdida es de la representación (modelo/composición), no del índice.
hits_exactos = aurum_search(texto_peor, top_k=10, exact=True)
mismo_resultado = [h["product_id"] for h in hits] == [h["product_id"] for h in hits_exactos]
print(f"\n¿Top-10 ANN == top-10 exacto? {mismo_resultado} -> "
      + ("la pérdida NO es del ANN: es de representación." if mismo_resultado
         else "parte de la pérdida procede del ANN."))

relevantes_no_vistos = sorted(relevant_set(juicios) - {h["product_id"] for h in hits})
print(f"Relevantes juzgados que no entran en el top-10: {len(relevantes_no_vistos)} "
      f"(p. ej. {relevantes_no_vistos[:4]})")

Peor consulta densa: 33633 · 'disfraz halloween talla grande hombre' · nDCG@10=0.000
 rank product_id                                                                  title  score relevancia
    1 B078NVQ217 I LOVE FANCY DRESS LTD Disfraz DE Trenchcoat Negro para Adultos - Tren 0.8850 no juzgado
    2 B07HHX8TNL tianxinxishop Disfraz de Jon Snow para Hombre Disfraz de Rey Knight Me 0.8830 no juzgado
    3 B00EM9ZVAY           ORION COSTUMES unisexo Primo Eso Halloween peliculas disfraz 0.8828 no juzgado
    4 B07NY97H69 Larga Capa de Vampiro Diablo de Terciopelo con Capucha para Disfraz de 0.8819 no juzgado
    5 B07JCK13L4                 WIDMANN-Mad Joker Disfraz, multicolor, (XS) (WDM08039) 0.8819 no juzgado
    6 B0753RX9VR                   WIDMANN-Disfraz para adultos Malefica, large (03703) 0.8818 no juzgado
    7 B004RZL7UG Wonder - Disfraz de mujer maravilla de halloween para mujer, talla XS  0.8808 no juzgado
    8 B07TG2VB9F LOLANTA Traje de Hombre de Hago de Oz para Niñas T

In [29]:
# === Caso 2 (índice ANN): consulta sensible con ef bajo ======================
idx_sensible = int(np.argmin(solapamientos_por_consulta[4]))
consulta_sensible = consultas_fidelidad[idx_sensible]
v = vectores_fidelidad[idx_sensible]
ids_exacto = ids_busqueda(v, exact=True)
ids_ef4 = ids_busqueda(v, hnsw_ef=4)
perdidos = [i for i in ids_exacto if i not in ids_ef4]
print(f"Consulta más sensible al ANN: {consulta_sensible!r}")
print(f"recall@10 con ef=4: {solapamientos_por_consulta[4][idx_sensible]:.2f} "
      f"| con ef={EF_PRODUCCION}: {solapamientos_por_consulta[128][idx_sensible]:.2f}")
if perdidos:
    posiciones = {pid: ids_exacto.index(pid) + 1 for pid in perdidos}
    print(f"IDs del oráculo que ef=4 pierde (posición exacta): {posiciones}")
    print("Atribución: pérdida del grafo HNSW con cola de candidatos insuficiente; "
          f"se corrige subiendo hnsw_ef (producción usa {EF_PRODUCCION}).")
else:
    print("Con esta construcción (m=16, ef_construct=128) ni siquiera ef=4 pierde vecinos: "
          "el coste del grafo está en latencia, no en recall (ver tabla de §8).")

Consulta más sensible al ANN: 'base tapizada 160x200 sin patas'
recall@10 con ef=4: 1.00 | con ef=128: 1.00
Con esta construcción (m=16, ef_construct=128) ni siquiera ef=4 pierde vecinos: el coste del grafo está en latencia, no en recall (ver tabla de §8).


In [30]:
# === Caso 3 (datos/filtros): grafías de marca y filtro exacto ================
marcas = catalog_full["brand"].dropna()
grafias = marcas.groupby(marcas.str.lower()).nunique()
multigrafia = grafias[grafias > 1]
print(f"Marcas con varias grafías en el catálogo: {len(multigrafia)} "
      f"(ej.: {dict(marcas[marcas.str.lower().isin(multigrafia.index[:3])].value_counts().head(6))})")

# Evidencia: filtrar por el campo crudo 'brand' con una grafía concreta pierde resultados.
def contar(campo, valor):
    return client.count(COLECCION, exact=True, count_filter=models.Filter(must=[
        models.FieldCondition(key=campo, match=models.MatchValue(value=valor))])).count

ejemplo_marca = "NIKE"
print(f"count(brand == 'Nike') = {contar('brand', 'Nike')}  |  "
      f"count(brand == 'NIKE') = {contar('brand', 'NIKE')}  |  "
      f"count(brand_norm == 'nike') = {contar('brand_norm', 'nike')}")
print("Atribución: es un problema de DATOS (grafías inconsistentes), no del motor; la capa de "
      "ingesta lo corrige materializando 'brand_norm' y filtrando sobre él (§7). En la entrega "
      "anterior, este detalle + filtrado post-hoc dejaba las 4 consultas filtradas vacías.")

Marcas con varias grafías en el catálogo: 66 (ej.: {'3 Pommes': np.int64(4), 'ACTECOM': np.int64(2), 'Alfa': np.int64(2), '3 pommes': np.int64(1), 'actecom': np.int64(1), 'ALFA': np.int64(1)})


count(brand == 'Nike') = 0  |  count(brand == 'NIKE') = 295  |  count(brand_norm == 'nike') = 295
Atribución: es un problema de DATOS (grafías inconsistentes), no del motor; la capa de ingesta lo corrige materializando 'brand_norm' y filtrando sobre él (§7). En la entrega anterior, este detalle + filtrado post-hoc dejaba las 4 consultas filtradas vacías.


In [31]:
# === Caso 4 (duplicados): el error más ajustado de la regla ==================
dev_dup["pred"] = (dev_dup["score_top1"] >= REGLA_DUP["umbral"]) & (dev_dup["margen"] >= REGLA_DUP["margen_min"])
errores_dup = dev_dup[dev_dup["pred"] != dev_dup["es_duplicado"]]
if len(errores_dup):
    caso = errores_dup.iloc[0]
    tipo = "falso positivo" if caso["pred"] else "falso negativo"
    print(f"{tipo}: {caso['incoming_id']} score={caso['score_top1']:.4f} margen={caso['margen']:.4f} "
          f"(umbral={REGLA_DUP['umbral']}, margen_min={REGLA_DUP['margen_min']})")
else:
    # Sin errores: mostramos el caso con menos holgura frente al umbral (riesgo operativo).
    dev_dup["holgura"] = (dev_dup["score_top1"] - REGLA_DUP["umbral"]).abs()
    caso = dev_dup.sort_values("holgura").iloc[0]
    print("La regla no comete errores en desarrollo; el caso con menos margen de seguridad es:")
    print(f"  {caso['incoming_id']}: score={caso['score_top1']:.4f} "
          f"(a {caso['holgura']:.4f} del umbral {REGLA_DUP['umbral']}), "
          f"duplicado real={caso['es_duplicado']}")
    print("Atribución: las altas 'casi duplicadas' (misma ficha con campos vaciados o sufijos de "
          "distribuidor) comprimen los scores cerca del umbral; con 14 casos de desarrollo el "
          "riesgo de sobreajuste del umbral es real y conviene recalibrar con más etiquetas.")

La regla no comete errores en desarrollo; el caso con menos margen de seguridad es:
  DEV-NEW-007: score=0.9080 (a 0.0013 del umbral 0.9093), duplicado real=False
Atribución: las altas 'casi duplicadas' (misma ficha con campos vaciados o sufijos de distribuidor) comprimen los scores cerca del umbral; con 14 casos de desarrollo el riesgo de sobreajuste del umbral es real y conviene recalibrar con más etiquetas.


### Taxonomía de los fallos analizados

| Caso | Capa responsable | Evidencia | ¿Quién puede corregirlo? |
|---|---|---|---|
| 1 · peor consulta dev | **Representación** | top-10 ANN == top-10 exacto, relevantes fuera del top-10 | modelo mayor / texto más rico / reranking; no lo arregla el índice |
| 2 · consulta sensible | **Índice ANN** | vecinos del oráculo perdidos con `ef=4`, recuperados con `ef≥64` | subir `hnsw_ef` (coste: latencia, medido en §8) |
| 3 · filtro de marca | **Datos / filtros** | `brand` con grafías múltiples; recuentos distintos por grafía | normalización en ingesta (`brand_norm`); no es un fallo del motor |
| 4 · duplicado límite | **Regla de negocio / datos** | scores comprimidos cerca del umbral con 14 ejemplos | más etiquetas de calibración, coste asimétrico explícito |

**Limitaciones conocidas**: (i) 8 consultas de desarrollo son pocas para diferencias finas entre configuraciones;
(ii) los juicios ESCI no cubren todo el corpus (productos "no juzgados" se tratan como relevancia 0, cota inferior
del nDCG real); (iii) la calibración de duplicados usa 14 casos; (iv) las latencias son de un único equipo local
sin red (coste de red real no medido).

## 15. Conclusiones y recomendación de negocio

*(Los valores concretos están en las tablas anteriores y en `results/metricas_desarrollo.json`.)*

**Qué queda implementado y demostrado**
- Base de datos vectorial **persistente** (Qdrant en Docker) con esquema explícito, ingesta por lotes del
  catálogo completo, reingesta idempotente y recuento verificado tras reconexión.
- Búsqueda semántica con contrato uniforme y **filtros ejecutados en el motor** (4/4 consultas filtradas válidas).
- Mutaciones aplicadas y verificadas **en la colección** (por ID y por búsqueda), con idempotencia demostrada.
- Métricas de ranking corregidas y medidas sobre el **catálogo completo**, con baseline léxico controlado.
- Fidelidad ANN medida contra oráculo exacto y configuración de producción elegida con esa evidencia.
- Regla de duplicados calibrada en desarrollo con candidatos de la base vectorial.

**Recomendación**: desplegar el motor denso multilingüe con `hnsw_ef=128` y el filtro de marca normalizado.
Siguiente paso de mayor retorno: un **reranker cross-encoder** sobre el top-50 del motor (ataca los fallos de
representación del Caso 1, la capa dominante de error) y ampliar los juicios de desarrollo para poder calibrar
sin riesgo de sobreajuste.

**Propuestas futuras (no implementadas)**: reranking neuronal, búsqueda híbrida (denso + BM25 con fusión RRF),
cuantización escalar para reducir memoria, y monitorización de deriva del catálogo.

## 16. Limpieza segura (opcional)

La celda siguiente **no se ejecuta por defecto**. Borra únicamente las colecciones de esta práctica
(nunca un borrado amplio) y solo si se activa la bandera explícitamente. Las credenciales/URL del motor
viven en `.env` (ver `.env.example`); este notebook no contiene secretos.

In [32]:
LIMPIAR = False  # cambiar a True solo para desmantelar la práctica
if LIMPIAR:
    for col in ["aurum_catalogo", "aurum_catalogo_smoke"]:
        if client.collection_exists(col):
            client.delete_collection(col)
            print("Colección borrada:", col)
else:
    print("Limpieza desactivada (LIMPIAR=False). Colecciones presentes:",
          [c.name for c in client.get_collections().collections])

Limpieza desactivada (LIMPIAR=False). Colecciones presentes: ['aurum_catalogo', 'aurum_catalogo_smoke']
